# Internal Dependencies
<br>  

### References
- [Analyze java package metrics in a graph database](https://joht.github.io/johtizen/data/2023/04/21/java-package-metrics-analysis.html)
- [Calculate metrics](https://101.jqassistant.org/calculate-metrics/index.html)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)

In [ ]:
import os
import pandas as pd
from neo4j import GraphDatabase

In [ ]:
import matplotlib.pyplot as plot
import squarify

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell 
# before starting jupyter notebook to provide the password for the user "neo4j". 
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD")))
driver.verify_connectivity()

In [ ]:
def get_cypher_query_from_file(cypherFileName):
    with open(cypherFileName) as file:
        return ' '.join(file.readlines())

In [ ]:
def query_cypher_to_data_frame(filename : str, limit: int = 10_000):
    cypher_query_template = "{query}\nLIMIT {row_limit}"
    cypher_query = get_cypher_query_from_file(filename)
    cypher_query = cypher_query_template.format(query = cypher_query, row_limit = limit)
    records, summary, keys = driver.execute_query(cypher_query)
    return pd.DataFrame([r.values() for r in records], columns=keys)

In [ ]:
def query_first_non_empty_cypher_to_data_frame(*filenames : str, limit: int = 10_000):
    """
    Executes the Cypher queries of the given files and returns the first result that is not empty.
    If all given file names result in empty results, the last (empty) result will be returned.
    By additionally specifying "limit=" the "LIMIT" keyword will appended to query so that only the first results get returned.
    """    
    result=pd.DataFrame()
    for filename in filenames:
        result=query_cypher_to_data_frame(filename, limit)
        if not result.empty:
            return result
    return result

In [ ]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [ ]:
# Pandas DataFrame Display Configuration
pd.set_option('display.max_colwidth', 300)

## Git History - File Count by Directory

In [ ]:
git_file_directories = query_cypher_to_data_frame("../cypher/GitLog/List_git_files_directories.cypher", limit=70).sort_values(by="fileCount", ascending=False)

In [ ]:
git_file_directories.head(30)

### TreeMap using Squarify

In [ ]:
figure, axis = plot.subplots(figsize=(20,20))
axis.set_axis_off()
axis.set_title('Directories with the number of contained files')
squarify.plot(
    sizes=git_file_directories.fileCount, 
    label=git_file_directories.directoryName,
    text_kwargs={'color':'white', 'fontsize':9, 'fontweight':'bold'},
    edgecolor="white", 
    linewidth=4,
    ax=axis
)
plot.show()

### TreeMap using Plotly

In [ ]:
import plotly.express as plotly_express

figure = plotly_express.treemap(
    git_file_directories, 
    path=['directoryParentName', 'directoryName'], 
    values='fileCount', 
    title='Directories by file count'
)
figure.update_traces(textinfo="label+value")
figure.update_layout(margin = dict(t=50, l=15, r=15, b=15))
figure.show()


## Git History - Directory Commit Statistics

In [ ]:
git_file_directories_with_commit_statistics = query_cypher_to_data_frame("../cypher/GitLog/List_git_files_directories_with_commit_statistics.cypher", limit=70).sort_values(by="fileCount", ascending=False)

In [ ]:
git_file_directories_with_commit_statistics.head(30)

### Directories by commit count (Plotly)

In [ ]:
import plotly.express as plotly_express

figure = plotly_express.treemap(
    git_file_directories_with_commit_statistics, 
    path=['directoryParentName', 'directoryName'], 
    values='commitCount', 
    title='Directories by commit count'
)
figure.update_traces(textinfo="label+value")
figure.update_layout(margin = dict(t=50, l=15, r=15, b=15))
figure.show()


### Directories by author count (Plotly)

In [ ]:
import plotly.express as plotly_express

figure = plotly_express.treemap(
    git_file_directories_with_commit_statistics, 
    path=['directoryParentName', 'directoryName'], 
    values='authorCount', 
    title='Directories by commit author count'
)
figure.update_traces(textinfo="label+value")
figure.update_layout(margin = dict(t=50, l=15, r=15, b=15))
figure.show()
